# Evaluation against cold-room experiments

This notebook prepares laboratory SSA measurements, runs the B21 and B25 model variants for isothermal and temperature-gradient experiments, and compares model estimates with observations using RMSE and bias summaries. The final section visualizes the evaluation results as box plots.

## Inputs and workflow
- Reads experimental records from `ssa_bibio.xlsx` and `SSA2014-2018_Hachikubo.xls`.
- Interpolates experiment time series to hourly steps and computes model trajectories.
- Summarizes model errors separately for isothermal and temperature-gradient cases, then plots the distributions.

Run the setup and data-preparation cells before the evaluation cells. Keep the spreadsheets in the notebook's working directory, or update the file paths in the loading cells.

In [ ]:
# %matplotlib inline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
import seaborn as sns

import function_SSA as core_ssa

### gradient

In [ ]:
name = "gradient"
file = 'data/ssa_bibio.xlsx'
df = pd.read_excel(file, sheet_name=name, header=0)

experiment = {}
for name, group in df.groupby(df.exp):
    if ('Wa' not in name) and ('B1' not in name):
    # if ('Wa' not in name):
    
        experiment[name] = group
        experiment[name]['dt'] = experiment[name]['time (h)'].diff()
        experiment[name]['dt'].iloc[0] = experiment[name]['time (h)'].iloc[0]
        experiment[name]

In [ ]:
len(experiment)

In [ ]:
def interpolate_experiment(df, dt = 1):
    # Interpolate the experiment to a specific dt
    # dt in hour
    density = df['density (kg/m3)']
    ssa = df['ssa (m2/kg)']
    time = df['time (h)'].values
    if time[-1] > 200*24:
        time_hourly = np.arange(0, 200*24 + dt , dt)
    else:
        time_hourly = np.arange(0, time[-1] + dt , dt)
    density_hourly = np.interp(time_hourly, time, density)
    ssa_hourly = np.interp(time_hourly, time, ssa)

    df_hourly = pd.DataFrame({'time' : time_hourly, 
                              'gradient' : df['tempe gradient'].values[0], 
                              'temperature' : df['tempe'].values[0],
                              'density' : density_hourly,
                              'ssa' : ssa_hourly,
                              'grain_type' : df['snow type'].values[0]})

    return df_hourly

experiment_hourly = {}
for name, df in experiment.items():
    experiment_hourly[name] = interpolate_experiment(df, dt = 1)

experiment_hourly

In [ ]:
sphericity_dict = {'PP' : 0.5, 'DF' : 0.5, 'RG' : 0.99, 'FC' : 0, 'DH' : 0, 'MF' : 0.99, 'IF' : 0.5, 'SH' : 0.5, 'PPgp' : 0.5}

def evolve_do(df_layer, dt, og = False, exponant = 0.4):
    #B21
    #dt in second
    do_toy = []
    for index, row in df_layer.reset_index().iterrows():
        if index == 0:
            s_0 = sphericity_dict[row.grain_type]
            d_0 = 6/(row.ssa*917)
            do_toy.append(d_0)
        elif index == 1 :
            s_f = np.clip(core_ssa.grad_s(row.temperature + 273.15, row.gradient, exponant = exponant) * dt * 3600 + s_0, a_min = 0, a_max = 1)
            do = core_ssa.grad_d(row.temperature + 273.15, row.gradient, row.density, s_f, d_0, og, exponant = exponant)* dt * 3600 + d_0
            do_toy.append(do)
        else:
            # print(core_ssa.grad_s(row.temperature + 273.15, row.gradient, exponant = exponant))
            s_f = np.clip(core_ssa.grad_s(row.temperature + 273.15, row.gradient, exponant = exponant)* dt * 3600 + s_f, a_min = 0, a_max = 1)
            # print(core_ssa.grad_d(row.temperature + 273.15, row.gradient, row.density, s_f, do, og, exponant = exponant))
            do = core_ssa.grad_d(row.temperature + 273.15, row.gradient, row.density, s_f, do, og, exponant = exponant)* dt * 3600 + do
            do_toy.append(do)

    return np.array(do_toy)


def evolve_ssa_B25(df_layer, dt, a_A = 1320, a_B = 1.52e-2, a_L = 1.09e-2, ssa_minimum = 0, a_tan = 0.3):
    #B25
    #dt in hour
    ssa_b25 = []
    for index, row in df_layer.reset_index().iterrows():
        if index == 0:
                ssa = row.ssa
                ssa_b25.append(row.ssa)
        else:
            ssa = core_ssa.ssa_B25(ssa, row.temperature + 273.15, row.gradient, row.density, dt, 
                                    a_A = a_A, 
                                    a_B = a_B, 
                                    a_L = a_L, 
                                    ssa_minimum = ssa_minimum, 
                                    a_tan = a_tan)
            ssa_b25.append(ssa)

    return np.array(ssa_b25)


In [ ]:
experiment_TG_hourly = {}
for key, df in experiment_hourly.items():
    if df.ssa.values[0] > 18:
        experiment_TG_hourly[key] = df

In [ ]:
len(experiment_TG_hourly)

In [ ]:
name = "iso"
file = 'data/ssa_bibio.xlsx'
df = pd.read_excel(file, sheet_name=name, header=0)

experiment_iso = {}
for name, group in df.groupby(df.exp):
    experiment_iso[name] = group
    experiment_iso[name]['dt'] = experiment_iso[name]['time (h)'].diff()
    experiment_iso[name]['dt'].iloc[0] = experiment_iso[name]['time (h)'].iloc[0]

experiment_iso_hourly = {}
for name, df in experiment_iso.items():
    experiment_iso_hourly[name] = interpolate_experiment(df, dt = 1)


In [ ]:
def interpolate_experiment_japan(df, dt = 1):
    # Interpolate the experiment to a specific dt
    # dt in hour
    ssa = np.array(df.ssa, dtype = float)
    time = np.array(df.time, dtype = float)
    if time[-1] > 200*24:
        time_hourly = np.arange(0, 200*24 + dt , dt)
    else:
        time_hourly = np.arange(0, time[-1] + dt , dt)

    ssa_hourly = np.interp(time_hourly, time, ssa)

    df_hourly = pd.DataFrame({'time' : time_hourly,
                              'density' : 50,
                              'gradient' : df.gradient.values[0], 
                              'temperature' : df.temperature.values[0] - 273.15,
                              'ssa' : ssa_hourly,
                              'grain_type' : 'PP'})

    return df_hourly

def ssa_2_do(ssa):
    return 6/(916.7*ssa)

def compute_rmse_experiment(experiment, model, do = True):
    l_rmse, l_bias = [], []
    for key, df in experiment.items():
        if df.ssa.values[0] > 18:
            if do:
                rmse = np.sqrt(np.mean((ssa_2_do(df[model]) - ssa_2_do(df.ssa))**2))
                bias = np.mean(ssa_2_do(df[model]) - ssa_2_do(df.ssa))
            else:
                rmse = np.sqrt(np.mean((df[model] - df.ssa)**2))
                bias = np.mean(df[model] - df.ssa)  

            l_rmse.append(rmse)
            l_bias.append(bias)

    return np.array(l_rmse), np.array(l_bias)

In [ ]:
df_japan_2013 = pd.read_excel('data/SSA2014-2018_Hachikubo.xls', sheet_name = 'main2013')

five_multiple = [i for i in range(len(df_japan_2013.columns)+1)][::5]

experiment_japan_2013 = {}
for i in range(len(five_multiple)):
    if i < len(five_multiple) - 1:
        df = df_japan_2013.iloc[:,five_multiple[i]:five_multiple[i+1]]
        name = '2013_' + df.columns[0]
        temperature = float(df.iloc[0,0][:3])
        columns = df.columns[0]
        df_clean = df.iloc[3:,:].reset_index()
        time = df_clean.iloc[:,2].values - df_clean.iloc[:,2].values[0]
        experiment_japan_2013[name] = pd.DataFrame({'time': time, 'ssa' :  df_clean.iloc[:,3], 'temperature' : temperature, 'gradient' :  0}).dropna()

df_japan_2014 = pd.read_excel('data/SSA2014-2018_Hachikubo.xls', sheet_name = 'main2014')

five_multiple = [i for i in range(len(df_japan_2014.columns)+1)][::5]

experiment_japan_2014 = {}
for i in range(len(five_multiple)):
    if i < len(five_multiple) - 1:
        df = df_japan_2014.iloc[:,five_multiple[i]:five_multiple[i+1]]
        name = '2014_' + df.columns[0]
        if ('B' in name) or ('E' in name) or 'F' in name:
            continue
        else:
            temperature = float(df.iloc[0,0][:3])
            columns = df.columns[0]
            df_clean = df.iloc[3:,:].reset_index()
            time = df_clean.iloc[:,2].values - df_clean.iloc[:,2].values[0]
            experiment_japan_2014[name] = pd.DataFrame({'time': time, 'ssa' :  df_clean.iloc[:,3], 'temperature' : temperature, 'gradient' :  0}).dropna()



experiment_japan_2013_hourly = {}
for key, df in experiment_japan_2013.items():
    experiment_japan_2013_hourly[key] = interpolate_experiment_japan(df)

experiment_japan_2014_hourly = {}
for key, df in experiment_japan_2014.items():
    experiment_japan_2014_hourly[key] = interpolate_experiment_japan(df)

In [ ]:
experiment_iso_hourly.update(experiment_japan_2013_hourly)
experiment_iso_hourly.update(experiment_japan_2014_hourly)

## Isothermal

In [ ]:
# popt_iso = [0.087033, 0.13861171]
popt_iso = [0.082378, 906.580521]

In [ ]:
dt = 1
for key, df in experiment_iso_hourly.items():
    ssa_b21 = 6/(917 * evolve_do(df, dt, og = True))
    df['ssa_model_B21'] = ssa_b21
    ssa_b25 = evolve_ssa_B25(df, dt)
    df['ssa_model_B25'] = ssa_b25
    ssa_b25 = evolve_ssa_B25(df, dt, a_B = popt_iso[0], a_L = popt_iso[1])
    df['ssa_model_B25_opt'] = ssa_b25
    # ssa_b25 = evolve_ssa_B25(df, dt, a_B = popt_iso_min[0], a_L = popt_iso_min[1], ssa_minimum = 8)
    # df['ssa_model_B25_opt_min'] = ssa_b25
    # ssa_b25 = evolve_ssa_B25_dendritic(df, dt, a_B = popt_iso_D[0], a_L = popt_iso_D[1], a_D = popt_iso_D[2], ssa_minimum = 8)
    # df['ssa_model_B25_opt_D'] = ssa_b25

In [ ]:
rmse_b21_iso, bias_b21_iso = compute_rmse_experiment(experiment_iso_hourly, 'ssa_model_B21')
rmse_b25_iso, bias_b25_iso = compute_rmse_experiment(experiment_iso_hourly, 'ssa_model_B25')
rmse_b25_iso_opt, bias_b25_iso_opt = compute_rmse_experiment(experiment_iso_hourly, 'ssa_model_B25_opt')

In [ ]:
rmse_ssa_b21_iso, bias_ssa_b21_iso = compute_rmse_experiment(experiment_iso_hourly, 'ssa_model_B21', do = False)
rmse_ssa_b25_iso, bias_ssa_b25_iso = compute_rmse_experiment(experiment_iso_hourly, 'ssa_model_B25', do = False)
rmse_ssa_b25_iso_opt, bias_ssa_b25_iso_opt = compute_rmse_experiment(experiment_iso_hourly, 'ssa_model_B25_opt', do = False)

###  TG

In [ ]:
# popt_TG_A_min = [700.48514641]
popt_TG_A_min = [551.09181215]

In [ ]:
dt = 1
for key, df in experiment_hourly.items():
    ssa_b21 = 6/(917 * evolve_do(df, dt, og = True))
    df['ssa_model_B21'] = ssa_b21
    ssa_b25 = evolve_ssa_B25(df, dt)
    df['ssa_model_B25'] = ssa_b25
    ssa_b25 = evolve_ssa_B25(df, dt, a_A = popt_TG_A_min[0], a_B = popt_iso[0], a_L = popt_iso[1], ssa_minimum = 8)
    df['ssa_model_B25_opt'] = ssa_b25



In [ ]:
rmse_b21_TG, bias_b21_TG = compute_rmse_experiment(experiment_TG_hourly, 'ssa_model_B21')
rmse_b25_TG, bias_b25_TG = compute_rmse_experiment(experiment_TG_hourly, 'ssa_model_B25')
rmse_b25_TG_opt, bias_b25_TG_opt = compute_rmse_experiment(experiment_TG_hourly, 'ssa_model_B25_opt')

In [ ]:
rmse_ssa_b21_TG, bias_ssa_b21_TG = compute_rmse_experiment(experiment_TG_hourly, 'ssa_model_B21', do = False)
rmse_ssa_b25_TG, bias_ssa_b25_TG = compute_rmse_experiment(experiment_TG_hourly, 'ssa_model_B25', do = False)
rmse_ssa_b25_TG_opt, bias_ssa_b25_TG_opt = compute_rmse_experiment(experiment_TG_hourly, 'ssa_model_B25_opt', do = False)

In [ ]:
print(f'B21 ET median SSA : {np.round(np.median(rmse_ssa_b21_iso),2)}, B25 ET median SSA  : {np.round(np.median(rmse_ssa_b25_iso_opt),2)}')
print(f'B21 TG median SSA : {np.round(np.median(rmse_ssa_b21_TG),2)}, B25 TG median SSA : {np.round(np.median(rmse_ssa_b25_TG_opt),2)}')

print(f'B21 ET median dopt : {np.round(np.median(rmse_b21_iso)/1e-3,2)}, B21 ET median dopt : {np.round(np.median(rmse_b25_iso_opt)/1e-3,2)}')
print(f'B21 TG median dopt  : {np.round(np.median(rmse_b21_TG)/1e-3,2)}, B25 TG median dopt : {np.round(np.median(rmse_b25_TG_opt)/1e-3,2)}')

### Box plot of result

In [ ]:
# 1. Box styling dictionaries
black_props = dict(color="black")
box_props = dict(edgecolor="black")

# 2. Initialize 2x2 subplot matrix
fig, ax = plt.subplots(2, 2, figsize=(5, 5))


# 3. Helper function to calculate positions and plot with pure matplotlib
def plot_pure_grouped_box(b21_data1, b25_data1, b21_data2, b25_data2, axis):
    # Consolidate all 4 data arrays for this subplot
    data = [b21_data1, b25_data1, b21_data2, b25_data2]

    # Calculate explicit X-coordinates to group them: [1.8, 2.2,   3.8, 4.2]
    # SSA group centered at 2, mm group centered at 4
    positions = [1.8, 2.2, 3.8, 4.2]

    # Generate the boxplot
    bp = axis.boxplot(
        data,#
        positions=positions,
        widths=0.35,
        patch_artist=True,
        boxprops=box_props,
        whiskerprops=black_props,
        capprops=black_props,
        medianprops=black_props,
    )

    # Apply colors alternatingly (B21 = dodgerblue, B25 = darkorange)
    bp["boxes"][0].set_facecolor("dodgerblue")
    bp["boxes"][1].set_facecolor("darkorange")
    bp["boxes"][2].set_facecolor("dodgerblue")
    bp["boxes"][3].set_facecolor("darkorange")

    # Format X-axis layout
    axis.set_xticks([2, 4])
    axis.set_xticklabels(["ET", "TG"])
    axis.grid(True, linestyle="--", alpha=0.5)

    return bp  # Return to extract legend elements later


# 4. Populate Row 0 (RMSE)
plot_pure_grouped_box(
    rmse_ssa_b21_iso,
    rmse_ssa_b25_iso_opt,
    rmse_ssa_b21_TG,
    rmse_ssa_b25_TG_opt,
    ax[0, 0],
)

# 5. Populate Row 1 (Bias)
plot_pure_grouped_box(
    bias_ssa_b21_iso,
    bias_ssa_b25_iso_opt,
    bias_ssa_b21_TG,
    bias_ssa_b25_TG_opt,
    ax[0, 1],
)

bp_sample = plot_pure_grouped_box(
    rmse_b21_iso * 1e3,
    rmse_b25_iso_opt * 1e3,
    rmse_b21_TG * 1e3,
    rmse_b25_TG_opt * 1e3,
    ax[1, 0],
)
plot_pure_grouped_box(
    bias_b21_iso * 1e3,
    bias_b25_iso_opt * 1e3,
    bias_b21_TG * 1e3,
    bias_b25_TG_opt * 1e3,
    ax[1, 1],
)

# 6. Global Labels & Layout Configuration
ax[0, 0].set_ylabel("RMSE (m$^2$ kg$^{-1}$)")
ax[0, 1].set_ylabel("Bias (m$^2$ kg$^{-1}$)")
ax[1, 0].set_ylabel("RMSE (mm)")
ax[1, 1].set_ylabel("Bias (mm)")
ax[0, 0].set_title("a) SSA")
ax[0, 1].set_title("b) SSA")
ax[1, 0].set_title("c) $d_{opt}$")
ax[1, 1].set_title("d) $d_{opt}$")
ax[0,1].axhline(y=0, color='k', linestyle='--', alpha = 0.6)
ax[1,1].axhline(y=0, color='k', linestyle='--', alpha = 0.6)

# 7. Add Global Bottom Legend using box colors
fig.legend(
    [bp_sample["boxes"][0], bp_sample["boxes"][1]],
    ["B21", "B25"],
    loc="lower center",
    ncols=2,
    bbox_to_anchor=(0.56, .93),
)

fig.tight_layout()
fig.show()
# fig.savefig('/home/jum002/public_html/ssa/cold_room_boxplot.png', dpi = 300)